# MCLDNN — 5-Class Baseline Training
**Dataset**: RML2016.10a · Classes: BPSK, QPSK, 8PSK, QAM16, QAM64  
**Model**: MCLDNN (TF2 / Keras 3)  
**Purpose**: Primary model for publications and future comparisons

### Kaggle Inputs Required
- Dataset: `rml2016-5class` → contains `RML2016.10a_5class.pkl`
- Dataset: `amr-repo` → contains this repository's source code  
  (OR use the git clone cell below)


In [ ]:
# ── CELL 1: Environment Setup ────────────────────────────────────────────────
import subprocess, sys

# Option A: Clone from GitHub (set your repo URL)
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/AMR.git'  # <-- update this
BRANCH      = 'main'
REPO_DIR    = '/kaggle/working/AMR'

!git clone --branch {BRANCH} --depth 1 {GITHUB_REPO} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print(f'Repo cloned to {REPO_DIR}')

# Option B: If repo is added as a Kaggle dataset (uncomment and comment Option A)
# sys.path.insert(0, '/kaggle/input/amr-repo')
# REPO_DIR = '/kaggle/input/amr-repo'

%cd {REPO_DIR}

In [ ]:
# ── CELL 2: Install / verify dependencies ────────────────────────────────────
!pip install -q pyyaml

import tensorflow as tf
import keras
print(f'TensorFlow: {tf.__version__}')
print(f'Keras:      {keras.__version__}')
print(f'GPU:        {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── CELL 3: Configure experiment ─────────────────────────────────────────────
import yaml, os

CONFIG_PATH = 'configs/exp_5class_baseline.yaml'

# Override dataset path to point to Kaggle input
with open(CONFIG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['path'] = '/kaggle/input/rml2016-5class/RML2016.10a_5class.pkl'

# Save modified config back (Kaggle working dir is writable)
MOD_CONFIG = '/kaggle/working/exp_5class_baseline_kaggle.yaml'
with open(MOD_CONFIG, 'w') as f:
    yaml.dump(cfg, f)

print('Config ready:')
print(yaml.dump(cfg, default_flow_style=False))

In [ ]:
# ── CELL 4: (Optional) Resume from previous checkpoint ───────────────────────
# Set RESUME_WEIGHTS to the path of a checkpoint from a previous Kaggle session.
# Leave as None for a fresh training run.

RESUME_WEIGHTS = None
# RESUME_WEIGHTS = '/kaggle/input/rml2016-checkpoints/5class_baseline_epoch120.weights.h5'

if RESUME_WEIGHTS:
    print(f'Will resume from: {RESUME_WEIGHTS}')
else:
    print('Starting fresh training run.')

In [ ]:
# ── CELL 5: Run Training ─────────────────────────────────────────────────────
resume_flag = f'--resume {RESUME_WEIGHTS}' if RESUME_WEIGHTS else ''
!python src/train.py --config {MOD_CONFIG} {resume_flag}

In [ ]:
# ── CELL 6: Inspect results & save checkpoint for next session ────────────────
import os, glob

EXP_DIR = 'experiments/5class_baseline'

# Show key output files
for root, dirs, files in os.walk(EXP_DIR):
    for fn in files:
        full = os.path.join(root, fn)
        size = os.path.getsize(full) / 1e6
        print(f'  {full}  ({size:.2f} MB)')

# Read test score
import csv
score_file = os.path.join(EXP_DIR, 'results', 'test_score.csv')
if os.path.exists(score_file):
    with open(score_file) as f:
        print('\nTest Score:')
        for row in csv.reader(f):
            print(' ', row)

In [ ]:
# ── CELL 7: Display key figures inline ───────────────────────────────────────
from IPython.display import Image, display
import glob

for fig_path in sorted(glob.glob('experiments/5class_baseline/figures/*.png'))[:6]:
    print(fig_path)
    display(Image(fig_path))

In [ ]:
# ── CELL 8: Record run metadata (update CHANGELOG manually after session) ─────
import subprocess

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
print(f'Git commit : {commit}')

import numpy as np, pickle
acc_path = 'experiments/5class_baseline/results/acc.dat'
if os.path.exists(acc_path):
    acc = pickle.load(open(acc_path, 'rb'))
    best_snr = max(acc, key=acc.get)
    print(f'Peak accuracy : {max(acc.values()):.4f} at SNR={best_snr} dB')
    print(f'All SNR acc   : {dict(sorted(acc.items()))}')

print('\n>> Upload experiments/5class_baseline/checkpoints/best_model.weights.h5')
print('   to Kaggle Dataset "rml2016-checkpoints" or GitHub Release.')
print(f'\n>> Add to CHANGELOG.md:')
print(f'   Git commit: {commit}')
print(f'   Kaggle notebook version: <check the notebook version tab>')